In [2]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [6]:
# 0) Mount Drive (skip if already mounted)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# ============================
# 1) CONFIG
# ============================
CSV_PATH = "/content/drive/MyDrive/traffic_data/dhaka_timeseries_2025-02-15_7d_15min.csv"  # adjust if needed
TARGET_COL = "travel_time_seconds"  # you can change to current_speed_kmh if you want
LOOKBACK = 12   # how many past steps (e.g. 12 * 15min = 3 hours)
HORIZON = 1     # predict 1 step ahead
BATCH_SIZE = 128
LSTM_EPOCHS = 10
LSTM_HIDDEN = 64
LSTM_LAYERS = 2
LR = 1e-3

In [8]:
# Optional: limit edges for speed while prototyping
MAX_EDGES_FOR_BASELINES = 1200  # e.g. 500 or None for all

In [9]:
# ============================
# 2) Load & basic preprocess
# ============================
print("Loading CSV...")
df = pd.read_csv(
    CSV_PATH,
    parse_dates=["timestamp"]   # no compression, normal CSV
)

# sort by edge + time
df = df.sort_values(["u", "v", "key", "timestamp"])

# Optional: limit number of edges for quick experiments
if MAX_EDGES_FOR_BASELINES is not None:
    unique_edges = df[["u", "v", "key"]].drop_duplicates().head(MAX_EDGES_FOR_BASELINES)
    df = df.merge(unique_edges, on=["u", "v", "key"], how="inner")
    df = df.sort_values(["u", "v", "key", "timestamp"])
    print(f"Filtered to {MAX_EDGES_FOR_BASELINES} edges, rows now: {len(df):,}")


# ============================
# 3) Build sliding windows per edge
# ============================
def build_datasets_per_edge(
    df,
    target_col: str,
    lookback: int,
    horizon: int
):
    """
    Returns:
      X_train, y_train, X_val, y_val, X_test, y_test  (all numpy arrays)

    Each X_* has shape [n_samples, lookback]
    Each y_* has shape [n_samples]
    """
    X_train, y_train = [], []
    X_val,   y_val   = [], []
    X_test,  y_test  = [], []

    grouped = df.groupby(["u", "v", "key"], sort=False)

    for (u, v, k), g in grouped:
        g = g.sort_values("timestamp")

        values = g[target_col].values.astype(np.float32)
        T = len(values)
        if T < lookback + horizon:
            continue

        # time-based split per edge
        train_end = int(T * 0.70)
        val_end   = int(T * 0.85)

        for t in range(lookback, T - horizon + 1):
            x_seq = values[t - lookback : t]
            y_val_future = values[t + horizon - 1]

            if t < train_end:
                X_train.append(x_seq)
                y_train.append(y_val_future)
            elif t < val_end:
                X_val.append(x_seq)
                y_val.append(y_val_future)
            else:
                X_test.append(x_seq)
                y_test.append(y_val_future)

    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_val   = np.array(X_val)
    y_val   = np.array(y_val)
    X_test  = np.array(X_test)
    y_test  = np.array(y_test)

    print("Windowed dataset shapes:")
    print("  X_train:", X_train.shape, " y_train:", y_train.shape)
    print("  X_val:  ", X_val.shape,   " y_val:",   y_val.shape)
    print("  X_test: ", X_test.shape,  " y_test:",  y_test.shape)

    return X_train, y_train, X_val, y_val, X_test, y_test


X_train, y_train, X_val, y_val, X_test, y_test = build_datasets_per_edge(
    df, TARGET_COL, LOOKBACK, HORIZON
)

if len(X_train) == 0:
    raise RuntimeError("No training samples were created. Check lookback/horizon or data size.")


# ============================
# 4) Metrics: RMSE, MAE, MAPE
# ============================
def regression_metrics(y_true, y_pred, prefix=""):
    y_true = np.array(y_true).astype(float)
    y_pred = np.array(y_pred).astype(float)

    mae = mean_absolute_error(y_true, y_pred)
    # Older sklearn: no 'squared' argument, so we do sqrt manually
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    # MAPE with small epsilon to avoid division by 0
    eps = 1e-3
    mape = np.mean(
        np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))
    ) * 100.0

    print(f"{prefix}MAE : {mae:.4f}")
    print(f"{prefix}RMSE: {rmse:.4f}")
    print(f"{prefix}MAPE: {mape:.2f}%")
    return mae, rmse, mape

# ============================
# 5) Random Forest baseline
# ============================
print("\n===== RandomForest Regressor baseline =====")

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print("RandomForest results (target =", TARGET_COL, ")")
rf_mae, rf_rmse, rf_mape = regression_metrics(y_test, y_pred_rf, prefix="RF   | ")


# ============================
# 6) LSTM baseline (PyTorch)
# ============================

print("\n===== LSTM baseline (PyTorch) =====")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---- MinMax scaling for LSTM ----
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train_2d = X_train.reshape(-1, LOOKBACK)
X_val_2d   = X_val.reshape(-1, LOOKBACK)
X_test_2d  = X_test.reshape(-1, LOOKBACK)

X_train_scaled = x_scaler.fit_transform(X_train_2d)
X_val_scaled   = x_scaler.transform(X_val_2d)
X_test_scaled  = x_scaler.transform(X_test_2d)

y_train_raw = y_train.reshape(-1, 1)
y_val_raw   = y_val.reshape(-1, 1)
y_test_raw  = y_test.reshape(-1, 1)

y_train_scaled = y_scaler.fit_transform(y_train_raw)
y_val_scaled   = y_scaler.transform(y_val_raw)
y_test_scaled  = y_scaler.transform(y_test_raw)

X_train_lstm = X_train_scaled.reshape(-1, LOOKBACK, 1)
X_val_lstm   = X_val_scaled.reshape(-1, LOOKBACK, 1)
X_test_lstm  = X_test_scaled.reshape(-1, LOOKBACK, 1)

y_train_lstm = y_train_scaled.astype(np.float32)
y_val_lstm   = y_val_scaled.astype(np.float32)
y_test_lstm  = y_test_scaled.astype(np.float32)


class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = SeqDataset(X_train_lstm, y_train_lstm)
val_ds   = SeqDataset(X_val_lstm,   y_val_lstm)
test_ds  = SeqDataset(X_test_lstm,  y_test_lstm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)


class LSTMRegressor(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        last_hidden = out[:, -1, :]      # [batch, hidden_dim]
        out = self.fc(last_hidden)       # [batch, 1]
        return out


model = LSTMRegressor(
    input_dim=1,
    hidden_dim=LSTM_HIDDEN,
    num_layers=LSTM_LAYERS,
    dropout=0.1
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def run_epoch(dataloader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    losses = []
    with torch.set_grad_enabled(is_train):
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            preds = model(X_batch)
            loss = criterion(preds, y_batch)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            losses.append(loss.item())

    return float(np.mean(losses)) if losses else np.nan


best_val_loss = np.inf
best_state = None

for epoch in range(1, LSTM_EPOCHS + 1):
    train_loss = run_epoch(train_loader, model, optimizer)
    val_loss   = run_epoch(val_loader,   model, optimizer=None)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = model.state_dict()

    print(f"Epoch {epoch:02d} | train_loss = {train_loss:.5f} | val_loss = {val_loss:.5f}")

if best_state is not None:
    model.load_state_dict(best_state)

# ---- Evaluate on test set ----
model.eval()
y_pred_scaled_list = []
y_true_scaled_list = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch)
        y_pred_scaled_list.append(preds.cpu().numpy())
        y_true_scaled_list.append(y_batch.cpu().numpy())

y_pred_scaled = np.vstack(y_pred_scaled_list)
y_true_scaled = np.vstack(y_true_scaled_list)

y_pred_lstm = y_scaler.inverse_transform(y_pred_scaled).flatten()
y_true_lstm = y_scaler.inverse_transform(y_true_scaled).flatten()

print("\nLSTM results (target =", TARGET_COL, ")")
lstm_mae, lstm_rmse, lstm_mape = regression_metrics(y_true_lstm, y_pred_lstm, prefix="LSTM | ")

print("\n===== SUMMARY =====")
print("RandomForest:")
print(f"  MAE : {rf_mae:.4f}")
print(f"  RMSE: {rf_rmse:.4f}")
print(f"  MAPE: {rf_mape:.2f}%")
print("LSTM:")
print(f"  MAE : {lstm_mae:.4f}")
print(f"  RMSE: {lstm_rmse:.4f}")
print(f"  MAPE: {lstm_mape:.2f}%")

Loading CSV...
Filtered to 1200 edges, rows now: 806,400
Windowed dataset shapes:
  X_train: (549600, 12)  y_train: (549600,)
  X_val:   (121200, 12)  y_val: (121200,)
  X_test:  (121200, 12)  y_test: (121200,)

===== RandomForest Regressor baseline =====
RandomForest results (target = travel_time_seconds )
RF   | MAE : 19.4676
RF   | RMSE: 64.7155
RF   | MAPE: 29.37%

===== LSTM baseline (PyTorch) =====
Using device: cuda
Epoch 01 | train_loss = 0.00031 | val_loss = 0.00021
Epoch 02 | train_loss = 0.00023 | val_loss = 0.00022
Epoch 03 | train_loss = 0.00023 | val_loss = 0.00021
Epoch 04 | train_loss = 0.00023 | val_loss = 0.00021
Epoch 05 | train_loss = 0.00022 | val_loss = 0.00021
Epoch 06 | train_loss = 0.00022 | val_loss = 0.00022
Epoch 07 | train_loss = 0.00022 | val_loss = 0.00022
Epoch 08 | train_loss = 0.00022 | val_loss = 0.00020
Epoch 09 | train_loss = 0.00022 | val_loss = 0.00022
Epoch 10 | train_loss = 0.00022 | val_loss = 0.00021

LSTM results (target = travel_time_seconds